## 🥇 Gold Layer

In the Gold Layer, we will perform the final transformations to prepare the data for analysis. This includes cleaning up the track names and audio names by removing any additional information that may be present after a " - " separator.

## Change the path to import modules from the parent directory

In [ ]:
import sys
sys.path.append("..")

## Importing necessary libraries

In [ ]:
from pyspark.sql.functions import *
from spark_session import get_spark
from pyspark.sql.types import *
from config import get_config
import pandas as pd

## Detect environment

In [ ]:
cfg = get_config()

## Change settings to display all columns in the DataFrame

In [ ]:
pd.set_option("display.max_columns", None)

## Intialize Spark session

In [ ]:
spark = get_spark("analytics-spotify-user-gold")

## Read silver table

In [ ]:
if cfg["environment"] == "Local":
    print("✅ Environment: Local")
    
    # Read JSON files into Spark DataFrame
    df_silver = pd.read_csv("../data/export/silver/streaming_history.csv")
    df_silver = spark.createDataFrame(df_silver)

elif cfg["environment"] == "Databricks":
    print("✅ Environment: Databricks")
    
    # Read silver table
    df_silver = spark.table(cfg["name_table_silver"])

In [ ]:
df_silver.show(5, truncate=False)

In [ ]:
df_gold = (
    df_silver
    .withColumn(
        "nm_track_name",
        split(col("nm_track_name"), " - ").getItem(0)
    )   
)

In [ ]:
df_gold = (
    df_silver
    .withColumn(
        "nm_audio_name",
        when(
            col("audio_type") == "Music",
            split(col("nm_audio_name"), " - ").getItem(0)
        ).otherwise(col("nm_audio_name"))
    )   
)

In [ ]:
if cfg["environment"] == "Local":
    print("✅ Environment: Local")
    
    # Export dataframe to CSV
    df_gold.toPandas().to_csv("../data/export/gold/streaming_history_user_spotify.csv", index=False)
    print("✅ Exported CSV to ../data/export/gold/streaming_history_user_spotify.csv")

elif cfg["environment"] == "Databricks":
    print("✅ Environment: Databricks")

    # Write to Delta Lake table
    df_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(cfg["name_table_gold"])
    print(f"✅ Loaded table: {cfg['name_table_gold']}")

In [ ]:
spark.stop()